# Project 1

Welcome to the first NLP1 project. The goal of the first tutorial was to get you familiar with text segmentation and simple text classification for topic detection.
Given the skills you have developed in the first week, you will now be applying them within a different dataset and task.

We are recommending a set of tasks and associated datasets; using those, you will be experimenting on problems introduced during the tutorials. You will be applying hands-on the experiments you have already explored in the last tutorial, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and analysing errors. 

The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

As the tutorials will be building upon your knowledge throughout the weeks, the task and dataset you choose now will be the ones you use in projects 2 and 3; and the ones you will be finally writing about in your final blog post.  

## Get to work!

The next section has a checklist to help you not lose track of what needs to be done. Here we have a more informal guide, with prompts and questions for you. 


1. [Choose a dataset](http://nlp-illc.github.io/nlp1/). Spend some time understanding how the dataset looks and some statistics of it (e.g. how many instances of each classification label we have?)
2. Remember what we learnt during the tutorial: to develop any classifier of the kind we saw, we needed a way to segment text. Do you want to use the tokeniser we have developed? Do you need to train a new one? If you do, what hyperparameters shall you use? What data shall you use to train it? Consider that your dataset may be too small to train a tokeniser! Think whether you will need a larger resource of similar nature to train a tokeniser.
3. Start thinking how to represent your documents into features. We have experimented with token occurences and frequencies. Can you think of any other ways, possibly? 
4. We had experimented with Logistic Regression and a shallow neural network. You can experiment with and assess different architectures now and their hyper parameters (e.g. add layers, possibly change the activation functions, drop out rates).
5. Evaluate your models! Was the evaluation method adequate for your new experimental setup? Do you want to assess other aspects of performance?
6. Analyse the various models you assessed and try to decide on which is the best.

Be mindful that you will be using your experiments to write up your final blog, so make sure they are clean and neat!

You can write your own code and you can reuse code from the tutorial and its packages or from other sources. You may use coding assistants if you like, but use it mindfully as you take full responsibility for this kind of (semi-)automated coding. Also, keep a record of the kinds of things you delegate to any form of assistant, as you will be asked to disclose this in your blog post (not because we will pass a value judgement on it, but because it is a widespread and good practice to document such things).

You may use jupyter notebooks or python files, it's up to you as we will not directly assess your code. But, once again, be organised, as we will ask you to share a reproducible version of your experiments along with your blog post (again, because this is how real research is publicised). 

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

## Checklist

What **we expect you to do**:

- [ ] Compare different systems of text classification for your choice of data/task.
- Your comparisons should cover:
    - [ ] the two representation spaces in the tutorial (token indicators and bag-of-words) and an additional one of your choice.
    - [ ] the two kinds of architectures in the tutorial (log-linear and single-hidden-layer MLP) and an additional one of your choice (such as a modification of the MLP).
- [ ] Your comparisons must be based on cross-validation and a relevant metric (accuracy or else).
- [ ] You should make use of Bayesian estimation to help you analyse results in relation to a region of practical equivalence (ROPE), and you can motivate the ROPE size yourself.

What you need to **bring to the oral assessment**:
* You will be asked questions about the points above, you can bring a slide deck with up to six slides (one for each of the topics/questions in the 'Get to work!' section); you _will not_ give a presentation, rather, these slides are to be thought of as "backup slides" that you can use to guide your answers to the questions the TA will ask you.
* You may also be asked questions about the tutorial, for those, you don't need to prepare any deliverable (a good way to prepare is to work through the tutorial and reflect on the questions there).

What you should **keep in mind** given that you will eventually write a blog post about your work throughout the course:
* Organise your code;
* Summarise your motivations and findings;
* Document your sources and any form of AI-assistance you use.

# Datasets

In [1]:
from datasets import load_dataset

# ds = load_dataset("fancyzhx/ag_news") 
ds = load_dataset("stanfordnlp/sst2")
ds

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [2]:
import numpy as np

# let's calculate class frequencies
labels = np.array(ds["train"]["label"])
labels_uniq, counts = np.unique(labels, return_counts=True)
labels_uniq, counts / len(labels)

(array([0, 1]), array([0.44217435, 0.55782565]))

# Tokeniser -- sentencepiece PBE

In [3]:
from io import BytesIO
import sentencepiece as spm

model = BytesIO()
spm.SentencePieceTrainer.train(
    sentence_iterator=iter(ds["train"]["sentence"]),
    model_writer=model,
    model_type="bpe",
    vocab_size=4_000
)
tokenizer = spm.SentencePieceProcessor(model_proto=model.getvalue())

encoded = ds["train"].map(
    lambda batch: {"token_ids": tokenizer.encode(batch["sentence"])},
    batched=True,
)

I0000 00:00:1793960106.810256 136758946 sentencepiece_trainer.cc:149] Starts training with : 
trainer_spec {
  input_format: 
  model_prefix: 
  model_type: BPE
  vocab_size: 4000
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  auto_character_coverage: 0
  vocabulary_output_piece_score: 1
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
}
normalizer_spec {
  name: nmt_nfkc
  add_dummy_prefix: 1
  remove_extra_w

# Encoder -- BoW, bool(BoW), TF-IDF

In [4]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

vectorizer = CountVectorizer(
    analyzer=lambda token_ids: token_ids,
    vocabulary={token_id: token_id for token_id in range(tokenizer.vocab_size())},
)
bow = vectorizer.transform(encoded["token_ids"])
bool_bow = bow.astype(bool)
tfidf_transformer = TfidfTransformer().fit(bow)
tfidf = tfidf_transformer.transform(bow)

representations = {
    "bool(BoW)": bool_bow,
    "BoW": bow,
    "TF-IDF": tfidf,
}

# CatBoost, ReLu MLP, LogisticRegression

use accuracy as the classes are ballanced. we ignore the fact that tokenizer was trained on full cv dataset as for our purposes it does not matter

In [5]:
import numpy as np
import pandas as pd
import torch
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from torch import nn
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

print(f"PyTorch device: {device}")

PyTorch device: mps


In [6]:
def fix_random(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def train_mlp(features, labels, epochs=5, batch_size=256, seed=42):
    fix_random(seed)

    input_dim = features.shape[1]
    num_classes = len(np.unique(labels))

    model = nn.Sequential(
        nn.Linear(input_dim, 128),
        nn.ReLU(),
        nn.Linear(128, num_classes),
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters())
    loss_fn = nn.CrossEntropyLoss()

    dataset = TensorDataset(
        torch.from_numpy(features.toarray()),
        torch.as_tensor(labels, dtype=torch.long),
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model.train()
    for _ in range(epochs):
        for inputs, targets in loader:

            inputs = inputs.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()

            loss = loss_fn(model(inputs), targets)
            loss.backward()

            optimizer.step()

    return model


@torch.inference_mode()
def mlp_probabilities(model, features, batch_size=256):
    model.eval()

    batches = []
    dataset = TensorDataset(torch.from_numpy(features.toarray()))
    loader = DataLoader(dataset, batch_size=batch_size)

    for (inputs,) in loader:
        inputs = inputs.to(device)
        batches.append(model(inputs).softmax(dim=1).cpu().numpy())

    return np.concatenate(batches)

In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for fold, (train_idx, held_out_idx) in enumerate(
    tqdm(cv.split(labels, labels), total=cv.get_n_splits(), desc="CV folds"), start=1
):
    y_train, y_held_out = labels[train_idx], labels[held_out_idx]

    for name, features in representations.items():
        X_train = features[train_idx].astype(np.float32)
        X_held_out = features[held_out_idx].astype(np.float32)

        logistic = LogisticRegression(
            max_iter=1_000
        ).fit(X_train, y_train)
        
        mlp = train_mlp(X_train, y_train)

        catboost = CatBoostClassifier(
            iterations=1_200,
            learning_rate=0.1,
            depth=6,
            loss_function="Logloss",
            random_seed=42,
            verbose=False,
            allow_writing_files=False,
        ).fit(X_train, y_train)

        predictions = {
            "Logistic regression": logistic.predict(X_held_out),
            "PyTorch MLP": mlp_probabilities(mlp, X_held_out).argmax(axis=1),
            "CatBoost": catboost.predict(X_held_out).ravel(),
        }
        for classifier, predicted in predictions.items():
            results.append({
                "Fold": fold,
                "Representation": name,
                "Classifier": classifier,
                "Accuracy": accuracy_score(y_held_out, predicted),
            })

fold_scores = pd.DataFrame(results)
fold_scores.groupby(["Representation", "Classifier"])["Accuracy"].agg(
    mean="mean", std="std"
).round(4)

CV folds:   0%|          | 0/5 [00:00<?, ?it/s]

mean     std
Representation Classifier                         
BoW            CatBoost             0.8797  0.0015
               Logistic regression  0.8686  0.0013
               PyTorch MLP          0.9047  0.0023
TF-IDF         CatBoost             0.8826  0.0022
               Logistic regression  0.8622  0.0022
               PyTorch MLP          0.8783  0.0024
bool(BoW)      CatBoost             0.8794  0.0019
               Logistic regression  0.8681  0.0014
               PyTorch MLP          0.9039  0.0021

In [12]:
import plotly.express as px

fig = px.box(
    fold_scores,
    x="Classifier",
    y="Accuracy",
    color="Representation",
    boxmode="group",
    points="all",
    hover_data=["Fold"],
    title="Accuracy across five CV folds",
    height=400,
)
fig.update_yaxes(tickformat=".1%")
fig.show()